循环对所有组做线性回归

In [ ]:
import csv
from pathlib import Path
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy import sparse
import harmonypy as hm

In [ ]:
def linreg_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    # y=0 时 x 截距：x0 = -b/a；取绝对值
    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs

In [ ]:
def build_linreg_result(df):
    """
    df 的行名格式：
    ctrl1_0_cluster0 / dn1_15_cluster2 这类
    列名为 m/z
    """
    tmp = df.copy()

    # 解析行名：样本号、时间、cluster
    parts = tmp.index.to_series().str.extract(
        r"^([A-Za-z]+\d+)_(\d+)_cluster(\d+)$"
    )

    tmp["_sample"] = parts[0]
    tmp["_time"] = parts[1].astype(int)
    tmp["_cluster"] = parts[2].astype(int)

    # m/z 列
    mz_cols = [c for c in tmp.columns if c not in ["_sample", "_time", "_cluster"]]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))

    result_rows = []

    # 按 sample + cluster 分组
    for (sample, cluster), sub in tmp.groupby(["_sample", "_cluster"], sort=False):
        sub = sub.sort_values("_time")

        # 这里 x 实际上就是 [0,15,30,45,60]
        x = sub["_time"].to_numpy(dtype=float)

        for mz in mz_cols:
            y = sub[mz].to_numpy(dtype=float)
            slope, intercept, r2, x0_abs = linreg_stats(x, y)

            result_rows.append({
                "sample": sample,
                "mz": float(mz),
                "cluster": int(cluster),
                "slope": slope,
                "intercept": intercept,
                "r2": r2,
                "x0_abs": x0_abs
            })

    result_df = pd.DataFrame(result_rows)

    # 排序：先 sample，再 cluster，再 mz
    parts2 = result_df["sample"].str.extract(r"^([A-Za-z]+)(\d+)$")
    result_df["_prefix"] = parts2[0]
    result_df["_sample_num"] = parts2[1].astype(int)

    result_df = result_df.sort_values(
        by=["_prefix", "_sample_num", "cluster", "mz"],
        kind="stable"
    ).drop(columns=["_prefix", "_sample_num"])

    result_df = result_df.reset_index(drop=True)
    return result_df

In [ ]:
# ===== 参数区：以后换组主要改这里 =====
PARAM_BASE_PATH = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/CAST_Kmeans_seperate2")
PARAM_INPUT_BASE_PATH = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/CAST_Kmeans_seperate2/metaIntensity")
PARAM_CLUSTER_NPZ = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Kmeans_seperate/all_clusters.npz")
PARAM_GROUPS = ["ctrl3", "dn3"]
PARAM_TIME_POINTS = ["0", "15", "30", "45", "60"]
PARAM_R2_THRESHOLD = 0.9
PARAM_CLEAN_INPUT_CSV = True


In [ ]:
# ===== 直接运行版：不使用 def；换组只改上面的 PARAM_GROUPS =====
cast_results = {}

for GROUP in PARAM_GROUPS:
    print(f"\n===== Running {GROUP} =====")
    data_dir = Path(PARAM_BASE_PATH) / GROUP
    data_dir.mkdir(parents=True, exist_ok=True)
    input_data_dir = Path(PARAM_INPUT_BASE_PATH) / GROUP
    
    if PARAM_CLEAN_INPUT_CSV:
        for tp in PARAM_TIME_POINTS:
            csv_file = input_data_dir / f"{tp}.csv"
            print(f"\nReading file: {csv_file.name}")
            df = pd.read_csv(csv_file, sep=";", comment="#")
            print(df.iloc[:5, :5])
            df.to_csv(csv_file, sep=";", index=False)
            print(f"Cleaned and overwritten: {csv_file.name}")

    group_list = []
    for tp in PARAM_TIME_POINTS:
        df = pd.read_csv(csv_file, sep=";", index_col=0, header=0)
        df_t = df.T.copy()
        duplicated_mz = df_t.columns[df_t.columns.duplicated()].unique().tolist()
        if duplicated_mz:
            print(
                f"{GROUP} {tp}.csv duplicated mz: {len(duplicated_mz)}; "
                f"examples: {duplicated_mz[:5]}"
            )
        else:
            print(f"{GROUP} {tp}.csv duplicated mz: 0")
        
        df_t = df_t.loc[:, ~df_t.columns.duplicated()]  # 去掉包里筛出来的重复 mz
        df_t.index = [f"{GROUP}_{tp}_pixel{i + 1}" for i in range(df_t.shape[0])]
        group_list.append(df_t)

    group_merged = pd.concat(group_list, axis=0)
    print(f"{GROUP}_merged shape:", group_merged.shape)
    print(group_merged.head())

    with np.load(PARAM_CLUSTER_NPZ) as data:
        cluster_key = f"{GROUP}_cluster"
        group_cluster = data[cluster_key]

    group_merged.insert(0, "cluster", group_cluster)

    group_mz_cols = [c for c in group_merged.columns if c != "cluster"]
    group_mz_cols = sorted(group_mz_cols, key=lambda x: float(x))
    group_merged = group_merged[["cluster"] + group_mz_cols]
    group_merged.to_csv(data_dir / f"{GROUP}Intensity_merged.csv", sep=";")

    group_tmp = group_merged.copy()
    group_tmp["group"] = group_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)

    group_cluster_mean = group_tmp.groupby(
        ["group", "cluster"],
        sort=False,
    )[group_mz_cols].mean()

    group_cluster_mean.index = [
        f"{sample_group}_cluster{cluster_id}"
        for sample_group, cluster_id in group_cluster_mean.index
    ]

    group_sort_df = group_cluster_mean.copy()
    group_parts = group_sort_df.index.to_series().str.extract(
        r"^([A-Za-z]+)(\d+)_(\d+)_cluster(\d+)$"
    )

    group_sort_df["_prefix"] = group_parts[0]
    group_sort_df["_sample_num"] = group_parts[1].astype(int)
    group_sort_df["_time"] = group_parts[2].astype(int)
    group_sort_df["_cluster"] = group_parts[3].astype(int)

    group_sort_df = group_sort_df.sort_values(
        by=["_prefix", "_sample_num", "_cluster", "_time"],#"_prefix"按字母顺序排序
        kind="stable",
    )

    group_cluster_mean = group_sort_df.drop(
        columns=["_prefix", "_sample_num", "_time", "_cluster"]
    )
    group_cluster_mean.to_csv(data_dir / f"{GROUP}_cluster_mean.csv", sep=";")
    print(group_cluster_mean.iloc[:10, :5])

    group_linreg_result = build_linreg_result(group_cluster_mean)
    group_linreg_result.to_csv(data_dir / f"{GROUP}_linreg_result.csv", index=False)
    print(group_linreg_result.head())

    group_r2_mean = (
        group_linreg_result
        .groupby(["sample", "mz"], as_index=False)["r2"]
        .mean()
        .rename(columns={"r2": "mean_r2"})
    )

    group_good = group_r2_mean[group_r2_mean["mean_r2"] >= PARAM_R2_THRESHOLD].copy()

    group_count = (
        group_good
        .groupby("sample", as_index=False)
        .size()
        .rename(columns={"size": f"n_metabolites_r2_ge_{PARAM_R2_THRESHOLD}"})
    )

    group_good.to_csv(data_dir / f"{GROUP}_good_r2_ge_{PARAM_R2_THRESHOLD}.csv", index=False)
    group_count.to_csv(data_dir / f"{GROUP}_count_r2_ge_{PARAM_R2_THRESHOLD}.csv", index=False)

    print(f"\n{GROUP}_good:")
    print(group_good.head())
    print(f"\n{GROUP}_count:")
    print(group_count)

    cast_results[GROUP] = {
        "merged": group_merged,
        "cluster_mean": group_cluster_mean,
        "linreg_result": group_linreg_result,
        "r2_mean": group_r2_mean,
        "good": group_good,
        "count": group_count,
    }
